In [ ]:
# Cell 1: Install packages (chạy khi cần)
!pip install -q -U trl transformers peft bitsandbytes accelerate datasets wandb==0.15.3 >/dev/null 2>&1
print("Install command executed (check output above).")

✅ Install command executed (check output above).


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q -U transformers peft accelerate datasets trl wandb==0.15.3

print("Packages installed")

import torch
import gc
import json

torch.cuda.empty_cache()
gc.collect()

print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Free memory: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

✅ Packages installed

CUDA: True
GPU: Tesla P100-PCIE-16GB
Free memory: 16.79 GB


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import HfApi, login
from huggingface_hub.utils import HfHubHTTPError
import wandb
import torch
import gc
import json
import os

# Cell 3: Configs (không in token trực tiếp)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
PROJECT_NAME = "PTMXH-qwen-nobel-kg-finetuning"
SAVE_LORA_DIR = "./qwen25-05B-instruct-nobel-kg-lora-v2"
MERGED_DIR = "./qwen25-05B-instruct-nobel-kg-merged-v2"
MERGED_REPO_ID = "trangpt666/qwen25-05B-instruct-nobel-kg-v2"
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
print("Configs set. (Tokens read from env variables.)")

2025-12-08 03:15:30.802939: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765163730.972777      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765163731.023283      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Configs set. (Tokens read from env variables.)


In [ ]:
# Cell 4: Login to HuggingFace + wandb (guarded)
from huggingface_hub import login
import wandb
try:
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY)
        wandb.init(project=PROJECT_NAME)
        print("wandb logged in")
    else:
        print("WANDB_API_KEY not set - skipping wandb login")
except Exception as e:
    print("wandb login failed:", e)
try:
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("HuggingFace login OK")
    else:
        print("HF_TOKEN not set - skipping HF login")
except Exception as e:
    print("HF login failed:", e)

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lengocquang2554 (quang123). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.23.1 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.15.3
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251208_031548-zv4v06w4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run curious-fog-21
wandb: ⭐️ View project at https://wandb.ai/quang123/PTMXH-qwen-Nobel-KG-finetuning
wandb: 🚀 View run at https://wandb.ai/quang123/PTMXH-qwen-Nobel-KG-finetuning/runs/zv4v06w4


✅ wandb logged in
✅ HuggingFace login OK


In [ ]:
# Cell 5: Tokenizer test + prompt template
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


INFERENCE_PROMPT_STYLE = """Bên dưới là một hướng dẫn mô tả một tác vụ, đi kèm với một thông tin đầu vào để cung cấp thêm ngữ cảnh.
Hãy viết một phản hồi để hoàn thành yêu cầu một cách phù hợp.

### Instruction:
Bạn là một chuyên gia trích xuất thông tin để xây dựng knowledge graph về các nhà đoạt giải Nobel.
Tôi sẽ cung cấp cho bạn schema của các entities và các loại relations cần trích xuất từ văn bản.
Từ schema và văn bản được cung cấp, hãy trích xuất tất cả các entities và relations liên quan đến nhà đoạt giải Nobel được đề cập trong văn bản và trả về dưới định dạng JSON.
Hãy trích xuất TẤT CẢ entities và relations từ văn bản một cách CHI TIẾT và ĐẦY ĐỦ nhất có thể.
QUAN TRỌNG:
- Đừng bỏ sót bất kỳ thông tin nào (tên người, tổ chức, địa điểm, giải thưởng, lĩnh vực...)
- Trích xuất CÀNG NHIỀU entities và relations CÀNG TÓT
- Chỉ trả về JSON với 2 fields: "entities" và "relations". KHÔNG bao gồm field "text" hoặc "name".

## Schema:
ENTITY TYPES:
- Person: Nobel laureate (main subject)
- Person_Non_Laureate: Other individuals (not laureates)
- Award: Nobel Prize, other prizes
- Organization: Universities, institutions
- Position: Job titles, roles
- Field: Scientific/academic fields
- Occupation: Professions
- Country: Nationalities
- Event: Historical events
- Location: Cities, regions
- Notable_Work: Discoveries, inventions, theories

RELATION TYPES:
- RECEIVED: Person → Award (won prize)
- IS_CITIZEN_OF: Person → Country (nationality)
- WORKS_AS: Person → Occupation (job type)
- WORKS_IN_FIELD: Person → Field (research area)
- EDUCATED_AT: Person → Organization (studied)
- EMPLOYED_BY: Person → Organization (worked for)
- IS_MEMBER_OF: Person → Organization (membership)
- HOLDS_POSITION: Person → Position (current role)
- IS_SPOUSE_OF: Person ↔ Person (married) or Person ↔ Person_non_laureate
- PARTICIPATED_IN: Person → Event (attended)
- FOUNDED: Person → Organization (established organization/school)
- CO_FOUNDED: Person → Organization (co-established with others)
- CO_DISCOVERED_WITH: Person ↔ Person or Person → Person_Non_Laureate (collaborated on discovery)
- DEVELOPED: Person → Notable_Work

### Văn bản về {}:
{}

### Kết quả trích xuất:
{}
"""
print("INFERENCE_PROMPT_STYLE defined.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

INFERENCE_PROMPT_STYLE defined.


In [6]:
hf_dataset = load_dataset("trangpt666/nobel_laureates_extraction")
train_raw = hf_dataset["train"]
val_raw = hf_dataset["validation"]
test_raw = hf_dataset["test"]

print(f"\nOriginal - Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}")

MAX_TOKENS = 1536

def filter_by_token_length(example):
    text = example.get("text", "")
    name = example.get("name", "")
    prompt = INFERENCE_PROMPT_STYLE.format(name, text, "")
    tokens = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    return len(tokens) <= MAX_TOKENS

train_raw = train_raw.filter(filter_by_token_length)
val_raw = val_raw.filter(filter_by_token_length)
test_raw = test_raw.filter(filter_by_token_length)

print(f"Filtered - Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}")

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/817 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/102 [00:00<?, ? examples/s]

...


Original - Train: 817, Val: 102, Test: 103


Filter:   0%|          | 0/817 [00:00<?, ? examples/s]

Filter:   0%|          | 0/102 [00:00<?, ? examples/s]

Filter:   0%|          | 0/103 [00:00<?, ? examples/s]

Filtered - Train: 817, Val: 102, Test: 103


In [ ]:
def has_valid_relations(example):
    """Ensure training samples have relations"""
    relations = example.get("relations", [])
    if isinstance(relations, str):
        try:
            relations = json.loads(relations)
        except:
            return False
    return len(relations) > 0

# Filter out samples without relations
train_raw = train_raw.filter(has_valid_relations)
print(f"After filtering samples with relations - Train: {len(train_raw)}")

# ===============================
# FORMAT DATA
# ===============================

def format_example(sample):
    name = sample.get("name", "")
    text = sample.get("text", "")
    entities = sample.get("entities", [])
    relations = sample.get("relations", [])

    if isinstance(entities, str):
        try:
            entities = json.loads(entities)
        except:
            entities = []

    if isinstance(relations, str):
        try:
            relations = json.loads(relations)
        except:
            relations = []

    output_json = json.dumps({
        "entities": entities,
        "relations": relations
    }, ensure_ascii=False, indent=2)

    # SỬA: Format trực tiếp output_json vào prompt
    formatted_text = INFERENCE_PROMPT_STYLE.format(name, text, output_json)
    formatted_text = formatted_text + tokenizer.eos_token

    return {"text": formatted_text}

train_dataset = train_raw.map(format_example, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(format_example, remove_columns=val_raw.column_names)
test_dataset = test_raw.map(format_example, remove_columns=test_raw.column_names)

print(f"\nDataset prepared - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Check the format of the first training sample
print("\n=== CHECKING FIRST TRAINING SAMPLE FORMAT ===")
print(train_dataset[0])

Filter:   0%|          | 0/817 [00:00<?, ? examples/s]

After filtering samples with relations - Train: 691


Map:   0%|          | 0/691 [00:00<?, ? examples/s]

Map:   0%|          | 0/102 [00:00<?, ? examples/s]

Map:   0%|          | 0/103 [00:00<?, ? examples/s]


Dataset prepared - Train: 691, Val: 102, Test: 103

=== CHECKING FIRST TRAINING SAMPLE FORMAT ===
{'text': 'Bên dưới là một hướng dẫn mô tả một tác vụ, đi kèm với một thông tin đầu vào để cung cấp thêm ngữ cảnh.\nHãy viết một phản hồi để hoàn thành yêu cầu một cách phù hợp.\n\n### Instruction:\nBạn là một chuyên gia trích xuất thông tin để xây dựng knowledge graph về các nhà đoạt giải Nobel.\nTôi sẽ cung cấp cho bạn schema của các entities và các loại relations cần trích xuất từ văn bản.\nTừ schema và văn bản được cung cấp, hãy trích xuất tất cả các entities và relations liên quan đến nhà đoạt giải Nobel được đề cập trong văn bản và trả về dưới định dạng JSON.\nHãy trích xuất TẤT CẢ entities và relations từ văn bản một cách CHI TIẾT và ĐẦY ĐỦ nhất có thể.\nQUAN TRỌNG: \n- Đừng bỏ sót bất kỳ thông tin nào (tên người, tổ chức, địa điểm, giải thưởng, lĩnh vực...)\n- Trích xuất CÀNG NHIỀU entities và relations CÀNG TÓT\n- Chỉ trả về JSON với 2 fields: "entities" và "relations". KHÔNG bao 

In [ ]:
torch.cuda.empty_cache()
gc.collect()

print("\n=== Loading Qwen2.5-0.5B ===")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Model loaded")
print(f"Memory allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"Memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ===============================
# IMPROVED LORA CONFIG - Better for complex tasks
# ===============================

lora_cfg = LoraConfig(
    r=8,  # Tăng lên để model học tốt hơn
    lora_alpha=16,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()



=== Loading Qwen2.5-0.5B ===


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded
Memory allocated: 0.99 GB
Memory free: 15.74 GB
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [ ]:
import torch
import gc
from transformers import TrainerCallback, EarlyStoppingCallback

# Clear cache callback
class ClearCacheCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()

    def on_evaluate(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()

sft_config = SFTConfig(
    output_dir="./qwen_nobel_KG_extraction_result_v2",

    dataset_text_field="text",
    max_length=1536,
    packing=False,

    # 🔥 Training - More epochs for better learning
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-4,  # Tăng learning rate
    num_train_epochs=10,  # Tăng epochs
    warmup_steps=50,     # Thêm warmup

    # Optimization
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_torch",

    per_device_eval_batch_size=1,
    eval_strategy="epoch",
    eval_accumulation_steps=1,

    save_strategy="epoch",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    remove_unused_columns=False,

    report_to="wandb",
)

# ===============================
# TRAINER
# ===============================

torch.cuda.empty_cache()
gc.collect()

print("\n=== Creating trainer ===")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[
        ClearCacheCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=3,  # Dừng nếu 3 epochs không cải thiện
            early_stopping_threshold=0.01  # Threshold để xem có cải thiện
        )
    ],
)

import torch
from functools import wraps

original_evaluation_loop = trainer.evaluation_loop

@wraps(original_evaluation_loop)
def evaluation_loop_no_grad(*args, **kwargs):
    with torch.no_grad():
        return original_evaluation_loop(*args, **kwargs)

trainer.evaluation_loop = evaluation_loop_no_grad

print(f"Memory before training: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"Memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")



=== Creating trainer ===


Adding EOS to train dataset:   0%|          | 0/691 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/691 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/691 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Memory before training: 1.01 GB
Memory free: 15.72 GB


In [ ]:
# TRAIN
# ===============================

print("\n=== Starting training ===")

try:
    train_result = trainer.train()

    print("\n=== TRAINING COMPLETED ===")
    print(f"Training loss: {train_result.training_loss:.4f}")
    print(f"Training time: {train_result.metrics['train_runtime']:.2f}s")

except RuntimeError as e:
    if "out of memory" in str(e):
        print("\nOOM Error! Trying emergency cleanup...")
        torch.cuda.empty_cache()
        gc.collect()
        raise
    else:
        raise

# SAVE MODEL
# ===============================

print(f"\n=== Saving to {SAVE_LORA_DIR} ===")
trainer.save_model(SAVE_LORA_DIR)
tokenizer.save_pretrained(SAVE_LORA_DIR)

if wandb.run is not None:
    wandb.log({
        "final_train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics["train_runtime"],
    })

print("\nTraining completed and model saved!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



=== Starting training ===


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.525100,0.539108,0.570654,1039092.000000,0.881845
2,0.475200,0.515564,0.512955,2078184.000000,0.885647
3,0.451100,0.511235,0.497757,3117276.000000,0.886866
4,0.436000,0.511819,0.486875,4156368.000000,0.886648
5,0.409500,0.516498,0.469137,5195460.000000,0.886301



=== TRAINING COMPLETED ===
Training loss: 0.5423
Training time: 4319.34s

=== Saving to ./qwen25-05B-instruct-nobel-kg-lora-v2 ===

✅ Training completed and model saved!


In [ ]:
# Cell 11: EVAL ON TEST SET (FIXED + Safe Cleanup)

import torch
import gc
from transformers import Trainer, TrainingArguments

print("\n=== EVALUATION ON TEST SET ===")

# ===============================
# SAFE CLEANUP MEMORY
# ===============================

# Clean up trainer nếu tồn tại
if 'trainer' in globals():
    del trainer
    print("Trainer deleted")
else:
    print("Trainer not found (might be first run or cell 10 failed)")

torch.cuda.empty_cache()
gc.collect()

print(f"Memory available: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ===============================
# TOKENIZE TEST SET
# ===============================

def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=1536,
        padding=False,
    )
    result["labels"] = result["input_ids"].copy()
    return result

test_dataset_processed = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test dataset"
)

print(f"Test dataset processed: {len(test_dataset_processed)} examples")

# ===============================
# EVAL CONFIG
# ===============================

eval_config = TrainingArguments(
    output_dir="./qwen_nobel_KG_extraction_result_v2",
    per_device_eval_batch_size=1,
    fp16=True,
    do_train=False,
    do_eval=True,
    remove_unused_columns=False,
)

# ===============================
# TRAINER FOR EVALUATION ONLY
# ===============================

trainer_eval = Trainer(
    model=model,
    args=eval_config,
    eval_dataset=test_dataset_processed,
    tokenizer=tokenizer,
)

# ===============================
# RUN EVALUATION
# ===============================

try:
    print("\nRunning evaluation on test set...")
    test_results = trainer_eval.evaluate(eval_dataset=test_dataset_processed)

    print("\nTest Results:")
    for key, value in test_results.items():
        if isinstance(value, float):
            print(f"   {key}: {value:.4f}")
        else:
            print(f"   {key}: {value}")

    # Calculate perplexity
    if "eval_loss" in test_results:
        perplexity = torch.exp(torch.tensor(test_results["eval_loss"])).item()
        print(f"   perplexity: {perplexity:.4f}")

    # Log to wandb if available
    try:
        if wandb.run is not None:
            wandb.log({
                "final_test_loss": test_results.get("eval_loss", 0),
                "test_perplexity": perplexity if "eval_loss" in test_results else 0,
            })
            print("\nResults logged to WandB")
    except:
        print("\nWandB not available for logging")

    print("\n✅ Evaluation completed successfully!")

except RuntimeError as e:
    if "out of memory" in str(e):
    print(f"\nOOM during eval!")
        print(f"   Current test size: {len(test_dataset_processed)}")
        print(f"   Try reducing test set or max_length")
        torch.cuda.empty_cache()
        gc.collect()
    else:
        raise e

except Exception as e:
    print(f"\n❌ Evaluation failed: {e}")
    raise

# ===============================
# CLEANUP
# ===============================

if 'trainer_eval' in locals():
    del trainer_eval
    torch.cuda.empty_cache()
    gc.collect()
    print(f"\nMemory after eval: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB free")


=== EVALUATION ON TEST SET ===
⚠️ Trainer not found (might be first run or cell 10 failed)
💾 Memory available: 15.63 GB


Tokenizing test dataset:   0%|          | 0/103 [00:00<?, ? examples/s]

/tmp/ipykernel_47/1266502360.py:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_eval = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


Test dataset processed: 103 examples

🔍 Running evaluation on test set...



📊 Test Results:
   eval_loss: 0.4995
   eval_model_preparation_time: 0.0162
   eval_runtime: 39.7852
   eval_samples_per_second: 2.5890
   eval_steps_per_second: 2.5890
   perplexity: 1.6480

✅ Results logged to WandB

✅ Evaluation completed successfully!

💾 Memory after eval: 15.63 GB free


In [ ]:
# Cell 12: MERGE LoRA + BASE MODEL + UPLOAD

import os
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import HfApi, login
from huggingface_hub.utils import HfHubHTTPError

print("\n=== MERGING LoRA + BASE MODEL ===")

# ===============================
# SAFE CLEANUP MEMORY
# ===============================

# ✅ Safe cleanup - không lỗi nếu biến chưa tồn tại
variables_to_delete = ['model', 'trainer_eval', 'trainer']

for var_name in variables_to_delete:
    if var_name in globals():
        del globals()[var_name]
        print(f"✅ {var_name} deleted")
    else:
        print(f"⚠️ {var_name} not found (skipping)")

torch.cuda.empty_cache()
gc.collect()
print(f"💾 Memory cleared: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB free")

# ===============================
# LOAD BASE MODEL
# ===============================

print(f"\n📥 Loading base model: {MODEL_NAME}")

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cpu",  # Load to CPU to save GPU memory
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    print("✅ Base model loaded")
except Exception as e:
    print(f"❌ Failed to load base model: {e}")
    raise

# ===============================
# LOAD LORA WEIGHTS
# ===============================

print(f"\n📥 Loading LoRA weights from: {SAVE_LORA_DIR}")

try:
    # Check if LoRA directory exists
    if not os.path.exists(SAVE_LORA_DIR):
        raise FileNotFoundError(f"LoRA directory not found: {SAVE_LORA_DIR}")

    merged_model = PeftModel.from_pretrained(base_model, SAVE_LORA_DIR)
    print("✅ LoRA weights loaded")
except Exception as e:
    print(f"❌ Failed to load LoRA weights: {e}")
    print(f"   Make sure Cell 10 (training) completed successfully")
    raise

# ===============================
# MERGE WEIGHTS
# ===============================

print("\n🔄 Merging LoRA with base model...")

try:
    merged_model = merged_model.merge_and_unload()
    print("✅ Weights merged successfully")
except Exception as e:
    print(f"❌ Failed to merge weights: {e}")
    raise

# ===============================
# SAVE MERGED MODEL
# ===============================

print(f"\n💾 Saving merged model to: {MERGED_DIR}")

try:
    os.makedirs(MERGED_DIR, exist_ok=True)
    merged_model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    print("✅ Merged model saved")
except Exception as e:
    print(f"❌ Failed to save merged model: {e}")
    raise

# ===============================
# CLEANUP BEFORE UPLOAD
# ===============================

print("\n🧹 Cleaning up memory before upload...")

del base_model, merged_model
torch.cuda.empty_cache()
gc.collect()

print(f"💾 Memory after merge: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB free")

# ===============================
# UPLOAD TO HUGGING FACE
# ===============================

print("\n=== UPLOADING TO HUGGING FACE ===")

try:
    api = HfApi()

    # Check if repo exists
    try:
        api.repo_info(MERGED_REPO_ID, repo_type="model")
        print(f"✅ Repo exists: {MERGED_REPO_ID}")
    except HfHubHTTPError:
        print(f"📝 Creating repo: {MERGED_REPO_ID}")
        api.create_repo(repo_id=MERGED_REPO_ID, repo_type="model", private=False)
        print(f"✅ Repo created")

    # Upload folder
    print(f"\n📤 Uploading from {MERGED_DIR} to {MERGED_REPO_ID}...")
    api.upload_folder(
        folder_path=MERGED_DIR,
        repo_id=MERGED_REPO_ID,
        repo_type="model",
    )

    print(f"\n🎉 SUCCESS! Model uploaded to:")
    print(f"   https://huggingface.co/{MERGED_REPO_ID}")

except Exception as e:
    print(f"\n❌ Upload failed: {e}")
    print(f"   You can manually upload from: {MERGED_DIR}")
    raise

# ===============================
# FINAL STATUS
# ===============================

print("\n" + "="*80)
print("✅ ALL STEPS COMPLETED!")
print("="*80)
print(f"\n📊 Summary:")
print(f"   LoRA weights: {SAVE_LORA_DIR}")
print(f"   Merged model: {MERGED_DIR}")
print(f"   HuggingFace: https://huggingface.co/{MERGED_REPO_ID}")
print(f"   Memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print("\n🚀 Ready for inference!")


=== MERGING LoRA + BASE MODEL ===
✅ model deleted
⚠️ trainer_eval not found (skipping)
⚠️ trainer not found (skipping)
💾 Memory cleared: 15.63 GB free

📥 Loading base model: Qwen/Qwen2.5-0.5B-Instruct
✅ Base model loaded

📥 Loading LoRA weights from: ./qwen25-05B-instruct-nobel-kg-lora-v2
✅ LoRA weights loaded

🔄 Merging LoRA with base model...
✅ Weights merged successfully

💾 Saving merged model to: ./qwen25-05B-instruct-nobel-kg-merged-v2
✅ Merged model saved

🧹 Cleaning up memory before upload...
💾 Memory after merge: 15.63 GB free

=== UPLOADING TO HUGGING FACE ===
📝 Creating repo: trangpt666/qwen25-05B-instruct-nobel-kg-v2
✅ Repo created

📤 Uploading from ./qwen25-05B-instruct-nobel-kg-merged-v2 to trangpt666/qwen25-05B-instruct-nobel-kg-v2...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 SUCCESS! Model uploaded to:
   https://huggingface.co/trangpt666/qwen25-05B-instruct-nobel-kg-v2

✅ ALL STEPS COMPLETED!

📊 Summary:
   LoRA weights: ./qwen25-05B-instruct-nobel-kg-lora-v2
   Merged model: ./qwen25-05B-instruct-nobel-kg-merged-v2
   HuggingFace: https://huggingface.co/trangpt666/qwen25-05B-instruct-nobel-kg-v2
   Memory free: 15.63 GB

🚀 Ready for inference!


## Infer

In [15]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

model_dir = "trangpt666/qwen25-05B-instruct-nobel-kg-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from: {model_dir}")
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.float16,
    device_map=device,
    trust_remote_code=True,
)
model.eval()

Loading model from: trangpt666/qwen25-05B-instruct-nobel-kg-v2


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
INFERENCE_PROMPT_STYLE = """Bên dưới là một hướng dẫn mô tả một tác vụ, đi kèm với một thông tin đầu vào để cung cấp thêm ngữ cảnh.
Hãy viết một phản hồi để hoàn thành yêu cầu một cách phù hợp.

### Instruction:
Bạn là một chuyên gia trích xuất thông tin để xây dựng knowledge graph về các nhà đoạt giải Nobel.
Tôi sẽ cung cấp cho bạn schema của các entities và các loại relations cần trích xuất từ văn bản.
Từ schema và văn bản được cung cấp, hãy trích xuất tất cả các entities và relations liên quan đến nhà đoạt giải Nobel được đề cập trong văn bản và trả về dưới định dạng JSON.
QUAN TRỌNG:
- Đừng bỏ sót bất kỳ thông tin nào (tên người, tổ chức, địa điểm, giải thưởng, lĩnh vực...)
- Trích xuất CÀNG NHIỀU entities và relations CÀNG TÓT
- Chỉ trả về JSON với 2 fields: "entities" và "relations". KHÔNG bao gồm field "text" hoặc "name".

## Schema:
ENTITY TYPES:
- Person: Nobel laureate (main subject)
- Person_Non_Laureate: Other individuals (not laureates)
- Award: Nobel Prize, other prizes
- Organization: Universities, institutions
- Position: Job titles, roles
- Field: Scientific/academic fields
- Occupation: Professions
- Country: Nationalities
- Event: Historical events
- Location: Cities, regions
- Notable_Work: Discoveries, inventions, theories

RELATION TYPES:
- RECEIVED: Person → Award (won prize)
- IS_CITIZEN_OF: Person → Country (nationality)
- WORKS_AS: Person → Occupation (job type)
- WORKS_IN_FIELD: Person → Field (research area)
- EDUCATED_AT: Person → Organization (studied)
- EMPLOYED_BY: Person → Organization (worked for)
- IS_MEMBER_OF: Person → Organization (membership)
- HOLDS_POSITION: Person → Position (current role)
- IS_SPOUSE_OF: Person ↔ Person (married) or Person ↔ Person_non_laureate
- PARTICIPATED_IN: Person → Event (attended)
- FOUNDED: Person → Organization (established organization/school)
- CO_FOUNDED: Person → Organization (co-established with others)
- CO_DISCOVERED_WITH: Person ↔ Person or Person → Person_Non_Laureate (collaborated on discovery)
- DEVELOPED: Person → Notable_Work

### Văn bản về {}:
{}

### Kết quả trích xuất:
"""


In [ ]:
print("\nLoading test dataset...")
hf_dataset = load_dataset("trangpt666/nobel_laureates_extraction")
test_dataset = hf_dataset["test"]

# 🔥 IMPROVED: Không filter, chỉ truncate text khi cần
num_test_samples = 5

# Lấy samples và truncate text nếu quá dài
test_samples_list = []
for i in range(min(num_test_samples, len(test_dataset))):
    sample = test_dataset[i]
    text = sample.get("text", "")

    # ✅ Truncate text nếu quá dài (giữ 800 chars đầu)
    if len(text) > 800:
        text = text[:800] + "..."
        sample["text"] = text

    test_samples_list.append(sample)

print(f"Selected {len(test_samples_list)} test samples")

# Convert to iterable
test_samples = test_samples_list

# ===============================
# INFERENCE
# ===============================

valid_json_count = 0
total_pred_entities = 0
total_pred_relations = 0
total_gt_entities = 0
total_gt_relations = 0

for idx, sample in enumerate(test_samples):
    print(f"\n{'='*80}")
    print(f"TEST SAMPLE {idx + 1}/{num_test_samples}")
    print(f"{'='*80}")

    name = sample.get("name", "")
    text = sample.get("text", "")

    print(f"\n📌 Name: {name}")
    print(f"📄 Text length: {len(text)} chars")
    print(f"📄 Text preview: {text[:150]}...")

    # 🔥 IMPROVED: Format đúng với training
    prompt = INFERENCE_PROMPT_STYLE.format(name, text)

    # 🔥 IMPROVED: Tăng max_length để không truncate input
    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
        truncation=True,
        max_length=1536,  # ✅ Tăng lên 1536 như training
    ).to(device)

    print(f"Input tokens: {inputs['input_ids'].shape[1]}")

    # 🔥 IMPROVED: Generation params - Tăng max_new_tokens để sinh nhiều hơn
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=768,     # ✅ Tăng từ 512 → 768 để có nhiều entities hơn
            temperature=0.2,        # ✅ Tăng từ 0.1 → 0.2 để đa dạng hơn
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,  # ✅ Giảm từ 1.2 → 1.15
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 🔥 FIXED: Dùng marker mới
    split_marker = "### JSON (trích xuất chi tiết và đầy đủ):"

    if split_marker in generated_text:
        result = generated_text.split(split_marker)[-1].strip()
    else:
        # Fallback: tìm JSON block
        result = generated_text[len(prompt):].strip()

    print(f"\n🤖 Generated tokens: {outputs.shape[1] - inputs['input_ids'].shape[1]}")
    print(f"🤖 Raw output length: {len(result)} chars")
    print(f"🤖 Output preview:\n{result[:500]}")

    # 🔥 IMPROVED: Better JSON parsing
    parsed = None
    try:
        # Remove markdown code blocks
        if "```json" in result:
            result = result.split("```json")[1].split("```")[0]
        elif "```" in result:
            result = result.split("```")[1].split("```")[0]

        result = result.strip()

        # Try to extract only JSON part
        if "{" in result and "}" in result:
            start = result.find("{")
            # Find last complete }
            open_count = 0
            end = start
            for i in range(start, len(result)):
                if result[i] == "{":
                    open_count += 1
                elif result[i] == "}":
                    open_count -= 1
                    if open_count == 0:
                        end = i + 1
                        break

            if end > start:
                result = result[start:end]

        parsed = json.loads(result)

    except json.JSONDecodeError as e:
        print(f"\n⚠️ JSON parse error: {e}")

        # 🔥 IMPROVED: Better auto-fix
        try:
            # Fix incomplete brackets
            open_braces = result.count("{")
            close_braces = result.count("}")
            open_brackets = result.count("[")
            close_brackets = result.count("]")

            if open_braces > close_braces:
                result += "}" * (open_braces - close_braces)
            if open_brackets > close_brackets:
                result += "]" * (open_brackets - close_brackets)

            # Remove trailing incomplete text
            if result.endswith(","):
                result = result[:-1]

            parsed = json.loads(result)
            print("✅ Fixed incomplete JSON!")

        except:
            print("❌ Cannot fix JSON")

    # Evaluate
    if parsed and isinstance(parsed, dict):
        valid_json_count += 1
        pred_entities = len(parsed.get('entities', []))
        pred_relations = len(parsed.get('relations', []))
        total_pred_entities += pred_entities
        total_pred_relations += pred_relations

        print(f"\n✅ Valid JSON!")
        print(f"   - Predicted Entities: {pred_entities}")
        print(f"   - Predicted Relations: {pred_relations}")

        if pred_entities > 0:
            print(f"   - Sample entities: {parsed['entities'][:3]}")
        if pred_relations > 0:
            print(f"   - Sample relations: {parsed['relations'][:3]}")
    else:
        print(f"\n❌ Invalid JSON - Cannot parse")

    # Ground truth
    gt_entities = sample.get("entities", [])
    gt_relations = sample.get("relations", [])
    if isinstance(gt_entities, str):
        try:
            gt_entities = json.loads(gt_entities)
        except:
            gt_entities = []
    if isinstance(gt_relations, str):
        try:
            gt_relations = json.loads(gt_relations)
        except:
            gt_relations = []

    total_gt_entities += len(gt_entities)
    total_gt_relations += len(gt_relations)

    print(f"\n📊 Ground Truth:")
    print(f"   - GT Entities: {len(gt_entities)}")
    print(f"   - GT Relations: {len(gt_relations)}")

# ===============================
# SUMMARY
# ===============================

print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")
print(f"Valid JSON outputs: {valid_json_count}/{num_test_samples} ({valid_json_count/num_test_samples*100:.1f}%)")

if valid_json_count > 0:
    print(f"\nAverage per sample (valid only):")
    print(f"   - Predicted Entities: {total_pred_entities/valid_json_count:.1f}")
    print(f"   - Predicted Relations: {total_pred_relations/valid_json_count:.1f}")

print(f"\nOverall average:")
print(f"   - Predicted Entities: {total_pred_entities/num_test_samples:.1f}")
print(f"   - GT Entities: {total_gt_entities/num_test_samples:.1f}")
print(f"   - Predicted Relations: {total_pred_relations/num_test_samples:.1f}")
print(f"   - GT Relations: {total_gt_relations/num_test_samples:.1f}")

if total_gt_entities > 0:
    entity_coverage = (total_pred_entities / total_gt_entities) * 100
    print(f"\nEntity coverage: {entity_coverage:.1f}%")
if total_gt_relations > 0:
    relation_coverage = (total_pred_relations / total_gt_relations) * 100
    print(f"Relation coverage: {relation_coverage:.1f}%")

print("\n✅ Inference completed!")


Loading test dataset...
Selected 5 test samples

TEST SAMPLE 1/5

📌 Name: Klaus von Klitzing
📄 Text length: 803 chars
📄 Text preview: Nobel Prize in Physics. In 1962, Klitzing passed the Abitur at the Artland-Gymnasium in Quakenbrück, Germany, before studying physics at the Braunschw...
Input tokens: 746

🤖 Generated tokens: 492
🤖 Raw output length: 1665 chars
🤖 Output preview:
{
  "entities": [
    {
      "name": "Klaus von Klitzing",
      "label": "Person"
    },
    {
      "name": "Nobel Prize in Physics",
      "label": "Award"
    },
    {
      "name": "Abitur",
      "label": "Position"
    },
    {
      "name": "Artland-Gymnasium",
      "label": "Organization"
    },
    {
      "name": "physics",
      "label": "Field"
    },
    {
      "name": "University of Technology",
      "label": "Organization"
    },
    {
      "name": "Gottfried Landwehr Chair"

✅ Valid JSON!
   - Predicted Entities: 10
   - Predicted Relations: 7
   - Sample entities: [{'name': 'Klaus von Kli